In [ ]:
import os
import yaml
import pandas as pd

# === CONFIG ===
PROJECTS_DIR = r"C:\Users\Admin\OneDrive\Education\Master of Info - Thesis\Config Files"  # desktop
OUTPUT_DIR = r"C:\GitHub\Android-Mobile-Apps"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "instrumentation_test_summary.csv")

# === CLASSIFICATION KEYWORDS ===
TEST_TYPES = {
    #'firebase_test_lab': ['firebase test', 'gcloud firebase test android run'],
    'firebase_Full':['gcloud firebase test android run'],
    'firebase_Compact':['Firebase-Test-Lab-Action'],
    'appcenter_test': ['appcenter test run', 'microsoft/appcenter-test-cli-action'],
    'browserstack_test': ['browserstack', 'browserstack/github-actions'],
    'GitHub_emulator_full': ['android-emulator-runner'],
    'GitHub_emulator_compact':['malinskiy/action-android/emulator-run-cmd'],
    'GitHub_emulator_manual':['create avd'],
    'GitHub_GMD':['cleanManagedDevices'],
    #'GitHub_gradle':['connectedReleaseAndroidTest','connectedcheck', 'connectedDebugAndroidTest','connectedAndroidTest'],
    'Unit_Test': ['gradlew test', './gradlew test', 'testDebugUnitTest', 'testReleaseUnitTest',
                    'test', 'run unit tests', 'run: test', 'npm test', 'yarn test'],
    'Other':['instrumentation','instrument']
    #'GitHub_avd':['adb', 'avdmanager']
    
}

# === RESULTS STRUCTURE ===
project_results = {}

# === DETECTION LOGIC ===
def detect_testing_types(yaml_text):
    # Remove commented lines
    uncommented_text = '\n'.join(
        line for line in yaml_text.splitlines()
        if not line.strip().startswith('#')
    ).lower()

    found = set()
    for label, keywords in TEST_TYPES.items():
        for kw in keywords:
            if kw.lower() in uncommented_text:
                found.add(label)
    return found


# === MAIN PARSER ===
def parse_yaml_file(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            raw = f.read().replace('\t', ' ')
            detected = detect_testing_types(raw)  # <== Move this BEFORE parsing
            content = yaml.safe_load(raw)  # Parse just to check for validity
            if not content:
                return {'types': detected, 'error': True}
            return {'types': detected, 'error': False}
    except Exception as e:
        return {'types': set(), 'error': True}


# === PROJECT SCANNER ===
for root, _, files in os.walk(PROJECTS_DIR):
    for file in files:
        if file.endswith(('.yml', '.yaml')):
            file_path = os.path.join(root, file)
#            print(f"📄 Scanning: {file_path}")

            filename = os.path.basename(file_path)
            parts = filename.split(".")
            project_name = (parts[1] if len(parts) > 2 else parts[0]).lower() # Between first and second dot

            result = parse_yaml_file(file_path)
#            print(f"→ Project: {project_name}, Test Types: {result['types'] or 'none'}, YAML Error: {result['error']}")

            if project_name not in project_results:
                project_results[project_name] = {'types': set(), 'errors': 0}

            project_results[project_name]['types'].update(result['types'])
            if result['error']:
                project_results[project_name]['errors'] += 1

# === EXPORT CSV ===
rows = []
for project, result in project_results.items():
    # Remove 'Other' if there's at least one other test type
    cleaned_types = result['types']
    if 'Other' in cleaned_types and len(cleaned_types) > 1:
        cleaned_types = cleaned_types - {'Other'}

    rows.append({
        'project': project,
        'test_types': ', '.join(sorted(cleaned_types)) if cleaned_types else 'none',
        'yaml_errors': result['errors']
    })

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)

# === SAVE TO DATAFRAME ===
Total_Projects = df.copy()

print(f"\n✅ Summary written to: {OUTPUT_CSV}")






# Part two - Make the list for all projects

import pandas as pd

PROJECTS_DIR = r"C:\Users\Admin\OneDrive\Education\Master of Info - Thesis\Config Files"  # desktop
OUTPUT_DIR = r"C:\GitHub\Android-Mobile-Apps"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "Total_Project_v1.0.csv")

#read from table above
df = Total_Projects.copy()

# 0 - Unit Test: True if any test_type include "Unit_Test"
df['Unit Test'] = df['test_types'].apply(
    lambda x: any(item.strip().startswith('Unit_Test') for item in x.split(','))
)


# 1 - Instrumentation Testing: True if test_type is not "none" nor only unit_test is detected
df['Instrumentation Testing'] = df['test_types'].apply(
    lambda x: x.strip().lower() != 'none' and not (
        len([t for t in x.split(',') if t.strip()]) == 1 and x.strip() == 'Unit_Test'
    )
)

# 2 - GitHub Action: True if any test_type starts with "GitHub"
df['GitHub Action'] = df['test_types'].apply(
    lambda x: any(item.strip().startswith('GitHub') for item in x.split(','))
)

# 3 - GitHub Action Type: include all GitHub-related test types
df['GitHub Action Type'] = df['test_types'].apply(
    lambda x: ', '.join([item.strip() for item in x.split(',') if item.strip().startswith('GitHub')])
)

# 4 - Third_Party: True if any test_type does not start with "GitHub" and is not "none" or "other" or "unit_test"
df['Third_Party'] = df['test_types'].apply(
    lambda x: any(
        not item.strip().startswith('GitHub') and item.strip().lower() not in ['none', 'other','unit_test']
        for item in x.split(',')
    )
)

# 5 - Third_Party_Name: list the test types that do not start with "GitHub" and are not "none" or "other" or "unit_test"
df['Third_Party_Name'] = df['test_types'].apply(
    lambda x: ', '.join([
        item.strip() for item in x.split(',')
        if not item.strip().startswith('GitHub') and item.strip().lower() not in ['none', 'other','unit_test']
    ])
)


# Save it back to CSV if needed

df.to_csv(OUTPUT_CSV, index=False)

print(f"\n✅ Summary written to: {OUTPUT_CSV}")


# Optional: assign to Total_Projects for in-memory use
Total_Projects = df





✅ Summary written to: C:\GitHub\Android-Mobile-Apps\instrumentation_test_summary.csv
